# Working with Subpopulations

Subpopulations let you define named cohorts of patients and compare their sequence frequency profiles independently from the full population.

**Prerequisites:** Run `01_getting_started.ipynb` first to create the `covid_35k.sqlite3` database.

**What you will learn:**
1. Creating a subpopulation from a list of patient IDs
2. Adding and removing patients from a subpopulation
3. Listing subpopulation members
4. Querying and filtering subpopulation-specific sequence frequencies
5. Comparing frequency profiles between two cohorts

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import tspmdb
import pandas as pd

DB_PATH = '../covid_35k.sqlite3'
db = tspmdb.TspmDB(DB_PATH)
print('Opened database:', DB_PATH)

In [ ]:
# Load all patient IDs into a list — we'll split them into two cohorts
all_patient_ids = db.population.patients(as_list=True)
print(f'Total patients in database: {len(all_patient_ids):,}')

# Split roughly in half for demonstration purposes
midpoint = len(all_patient_ids) // 2
cohort_a_ids = all_patient_ids[:midpoint]
cohort_b_ids = all_patient_ids[midpoint:]
print(f'Cohort A: {len(cohort_a_ids):,} patients')
print(f'Cohort B: {len(cohort_b_ids):,} patients')

## 2. Creating Subpopulations

`db.subpopulation.create(subpop_id, patient_ids, description)` creates a named cohort and returns a `SubpopulationInstance`.

- `subpop_id` must be unique in the database. Passing `destructive=True` overwrites an existing subpopulation with the same ID.
- Pass `[]` as `patient_ids` to create an empty subpopulation and add patients later.

In [ ]:
# Create two cohorts
cohort_a = db.subpopulation.create(
    'cohort_a',
    cohort_a_ids,
    'First half of the COVID dataset patients',
    destructive=True
)

cohort_b = db.subpopulation.create(
    'cohort_b',
    cohort_b_ids,
    'Second half of the COVID dataset patients',
    destructive=True
)

print('Subpopulations created: cohort_a, cohort_b')

## 3. Listing Subpopulations

In [ ]:
# List all subpopulations defined in the database
subpops = db.subpopulation.list()
print('Existing subpopulations:')
for sp in subpops:
    print(' -', sp)

## 4. Managing Subpopulation Members

Use `.patients.list()`, `.patients.add()`, and `.patients.remove()` to manage which patients belong to a subpopulation.

In [ ]:
# List current members of cohort_a (first 5)
members = cohort_a.patients.list()
print(f'Cohort A has {len(members):,} patients')
print('First 5 members:', members[:5])

In [ ]:
# Retrieve an existing subpopulation from a fresh reference (simulates a new session)
cohort_a_ref = db.subpopulation.get('cohort_a')
print('Retrieved cohort_a, member count:', len(cohort_a_ref.patients.list()))

In [ ]:
# Remove the first patient from cohort_a
patient_to_remove = members[0]
cohort_a.patients.remove(patient_to_remove)
print(f'Removed {patient_to_remove} from cohort_a')
print(f'Cohort A now has {len(cohort_a.patients.list()):,} patients')

In [ ]:
# Add the patient back
cohort_a.patients.add(patient_to_remove)
print(f'Re-added {patient_to_remove} to cohort_a')
print(f'Cohort A now has {len(cohort_a.patients.list()):,} patients')

## 5. Querying Subpopulation Frequencies

`cohort.sequences.frequencies()` calculates frequency statistics on-the-fly from the `sequences` table, restricted to only the patients in the subpopulation.

- `observation_cnt` = total occurrences of the sequence across all subpopulation patients
- `patient_cnt` = number of distinct patients who have the sequence

The filtering parameters (`observation1`, `observation2`) and return options (`as_pandas`, `as_iterator`, `with_ids`) work the same as `db.population.frequencies()`.

In [ ]:
# Get all frequencies for cohort_a
freq_a = cohort_a.sequences.frequencies(as_pandas=True)
print(f'Cohort A frequency rows: {len(freq_a):,}')
freq_a.sort_values('patient_cnt', ascending=False).head(10)

In [ ]:
# Filter cohort_a frequencies by a specific obs_code_1
top_code = freq_a.sort_values('patient_cnt', ascending=False).iloc[0]['obs_code_1']
print('Filtering by obs_code_1:', top_code)

filtered_a = cohort_a.sequences.frequencies(observation1=top_code, as_pandas=True)
filtered_a.sort_values('patient_cnt', ascending=False).head(10)

## 6. Comparing Two Cohorts

A key use case is comparing the frequency profiles of two subpopulations to identify which sequences are more common in one group vs. another.

In [ ]:
# Get frequencies for both cohorts
freq_a = cohort_a.sequences.frequencies(as_pandas=True)
freq_b = cohort_b.sequences.frequencies(as_pandas=True)

# Create a merge key from the three identifying columns
key_cols = ['obs_code_1', 'obs_code_2', 'temporal_distance']

# Merge on the key columns, keeping all sequences from either cohort
comparison = freq_a.merge(
    freq_b,
    on=key_cols,
    how='outer',
    suffixes=('_cohort_a', '_cohort_b')
).fillna(0)

# Compute the difference in patient count
comparison['patient_cnt_diff'] = comparison['patient_cnt_cohort_a'] - comparison['patient_cnt_cohort_b']

print(f'Total unique sequences across both cohorts: {len(comparison):,}')
comparison.sort_values('patient_cnt_diff', ascending=False).head(10)

In [ ]:
# Sequences strongly overrepresented in cohort_b compared to cohort_a
comparison.sort_values('patient_cnt_diff', ascending=True).head(10)

## 7. Close the Database

In [ ]:
db.close()
print('Database closed.')